### 2- سیستم برای تشخیص دسته بندی کالا از روی عکس  

## imports

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
data_path = os.getenv('DATA_PATH')
products = pd.read_csv(f'{data_path}/BaSalam.products.csv')

In [ ]:
map_category = pd.read_csv(f"{data_path}/category_map.csv")
map_category.head()
products_ = pd.merge(products, map_category, left_on='categoryTitle', right_on='sub_category', how='left')
products_[['categoryTitle','main_category']].sample()
products_['main_category'].fillna('سایر', inplace=True)
products_.loc[products_['main_category']=='سلامت، درمان و طب', 'main_category'] = 'سلامت درمان و طب'

In [ ]:
products_[products_['main_category']=='سایر']['categoryTitle'].unique().tolist()

In [ ]:
products_['main_category'].value_counts()

## download images

In [33]:
sample = products_.sample(100_000, random_state=42)

In [ ]:
import os
import requests
import pandas as pd
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_fixed

# Create a base directory for saving images
base_dir = 'downloaded_images'
os.makedirs(base_dir, exist_ok=True)

# List to store image metadata
image_metadata = []
images_downloaded = 0  # Counter for downloaded images

# Function to download and save an image with retry
@retry(stop=stop_after_attempt(3), wait=wait_fixed(2))
def download_image(url, save_path):
    response = requests.get(url, timeout=10)
    if response.status_code == 200:
        with open(save_path, 'wb') as f:
            f.write(response.content)
    else:
        response.raise_for_status()

# Iterate through the sample dataframe with progress bar
total_images = len(sample)  # Total number of images
with tqdm(total=total_images, desc="Downloading Images", unit="img") as pbar:
    for index, row in sample.iterrows():
        category = row['main_category']
        sub_category = row['categoryTitle']
        product_id = row['_id']
        image_url = row['photo_MEDIUM']
        image_id = str(row['_id'])  # Ensure string format
        
        # Create a category-specific directory
        category_dir = os.path.join(base_dir, category)
        os.makedirs(category_dir, exist_ok=True)
        
        # Define the path to save the image
        image_filename = f'{image_id}.jpg'
        image_path = os.path.join(category_dir, image_filename)
        
        # Download and save the image
        try:
            download_image(image_url, image_path)
            images_downloaded += 1  # Increment counter on success
            # Add metadata only if download succeeds
            image_metadata.append({
                'product_id': product_id,
                'image_id': image_id,
                'image_path': image_path,
                'category': category,
                'sub_category': sub_category
            })
        except Exception as e:
            print(f"Failed to download {image_url}: {e}")
        
        pbar.update(1)  # Update progress bar

# Create DataFrame from metadata and save as CSV
if image_metadata:
    df_metadata = pd.DataFrame(image_metadata)
    csv_path = os.path.join(base_dir, 'metadata.csv')
    df_metadata.to_csv(csv_path, index=False) 
    print(f'\nMetadata CSV created at {csv_path}')
    print(f'Total images downloaded: {images_downloaded}')  # Print total count
else:
    print('\nNo images were downloaded, CSV not created')


Failed to download https://statics.basalam.com/public-16/users/JWo8AQ/01-16/Skh2w7Tcf6R6bqDaozMi40R21tonL9uKBq8OfKBikMJ617xfSh.jpg_512X512X70.jpg: RetryError[<Future at 0x1888e4f9280 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-16/users/MYwPn/12-27/jf4nhRZAKa0kMOtK4UphaO7vR2FEnUFbhsPokCFxHfXAOMffd1.jpg_512X512X70.jpg: RetryError[<Future at 0x1888e6ab140 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-17/users/MzZDq/03-06/2mcNykd5KBL2cbw0ODvdrq9cHy07souBAaiWzfYtc5tLPZvjsy.jpg_512X512X70.jpg: RetryError[<Future at 0x188bddc1fd0 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-16/users/3VP11/01-04/61VmE0JnIWTwlLguuqb1JGLnaXfVk2hlXribvBcmGdxrcW4zay.jpg_512X512X70.jpg: RetryError[<Future at 0x18924b46960 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-3/users/JWrq3Q/11-26/271BzJ1YK9ANZnankuGy6BEUTKog66DhviX1K0trzvnIt85AgW.jpg_512X512X70.jpg: RetryError[<Future at 0x1888da09490 state=finished raised HTTPError>]


...

Failed to download https://statics.basalam.com/public-18/users/0VEeXM/03-24/V6cG0kKUsZjjpJNK2BDZdHFXFkGbKlizi09TE3TJfETr5E9SqV.jpg_512X512X70.jpg: RetryError[<Future at 0x18924a70950 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-17/users/gqBnm7/03-03/4tN5uRU9TOnb6DhcN3kyq4WT5lGnLrDHTDRnXkfirMaqtMAiHZ.jpg_512X512X70.jpg: RetryError[<Future at 0x18924a72630 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-13/users/wd7Bm/10-19/TTda45O5LPAV6gfyJqbfBKv4IqxlMXj9hvPVJzAH8V2Oio5p2s.jpg_512X512X70.jpg: RetryError[<Future at 0x189459937d0 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-14/users/EZJLn/11-29/1f8Ic6IsWu7TLdGVmDDJs0M269A9BmxVviN3KCkjZl86pB0cSw.jpg_512X512X70.jpg: RetryError[<Future at 0x18924a25d90 state=finished raised HTTPError>]


Failed to download https://statics.basalam.com/public-7/users/49mnk/2105/vDJ7VNoUyv5tYuhFZy15MeQMh0Pilnn8jP35vD7z.jpeg_512X512X70.jpeg: RetryError[<Future at 0x18924a269c0 state=finished raised HTTPError>]
